## Agentic AI — Day 6
## Build a Complete Agent with LangChain + ChromaDB Memory

We will build the agent in **3 steps (milestones)**. Each milestone adds one new idea on top of the last.

| Milestone | What we add |
|---|---|
| 1 | A basic ReAct agent with tools (weather + calculator) |
| 2 | A knowledge search tool (look up office facts) |
| 3 | **ChromaDB memory** — the agent remembers what you said before |

---

### Before we start — What are LangChain and ChromaDB?

**LangChain** is a toolkit that makes it easier to build AI apps. Instead of writing everything yourself, LangChain gives you ready-made pieces — like a `@tool` decorator that tells the model how to call your Python functions.

**ChromaDB** is a small database that stores text as vectors (lists of numbers). When you ask it a question, it finds the stored text that *means something similar* — even if the words are different.

Think of it like this:
```
You say:   "What city does my team work in?"
ChromaDB finds:  "I manage the Pune team"   ← similar meaning, different words
Agent answers:   "Your team is in Pune"
```

---

## Step 0 — Install and Import

In [ ]:
# Run this once to install everything we need
!pip install  langchain langchain-community langchain-ollama chromadb

  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.42.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-manylinux_2_34_x86_64.whl.metadata (10 kB)
  Using cached kubernetes-36.0.2-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached mmh3-5.2.1-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (14 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached opentelemetry_exporter_otlp_proto_common-1.42.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached opentelemetry_proto-1.42.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached protobuf-6.3

In [3]:
%pip install langchain-classic


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import ast
import operator
import ollama

# LangChain imports
from langchain.tools import tool                        # turns a Python function into an AI tool
from langchain_ollama import OllamaEmbeddings           # converts text to vectors
from langchain_community.vectorstores import Chroma     # the vector database
from langchain_classic.memory import VectorStoreRetrieverMemory # memory backed by ChromaDB

# Which models to use — change if yours are different
MODEL       = "llama3.2:3b-instruct-q8_0"         # the chat model
EMBED_MODEL = "nomic-embed-text" # the embedding model  (ollama pull nomic-embed-text)

# A simple system prompt — tells the agent its job
SYSTEM = (
    "You are a helpful assistant for the Trinity office. "
    "Use tools for weather, math, and office facts. "
    "Answer once you have what you need."
)

print("✅ Everything imported successfully!")

✅ Everything imported successfully!


---
## Milestone 1 — A Basic Agent with Tools

We define two tools using LangChain's `@tool` decorator, then build a simple ReAct loop.

The `@tool` decorator does two things:
1. Reads the docstring to build a description the model can understand
2. Wraps the function so LangChain can manage it

### Define the tools

In [11]:
# ── Tool 1: Weather ───────────────────────────────────────────────────────────
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The city name, for example 'Pune'.
    """
    # Fake data for the demo — pretend this calls a real weather API
    data = {
        "pune":      "31°C, clear sky",
        "mumbai":    "33°C, humid",
        "delhi":     "29°C, hazy",
        "bangalore": "26°C, light rain",
    }
    return data.get(city.lower(), f"No weather data for {city}")


# ── Tool 2: Calculator ────────────────────────────────────────────────────────
# Helper: safely evaluate a math expression (no eval() security risk)
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}
def _eval_expr(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPS[type(node.op)](_eval_expr(node.left), _eval_expr(node.right))
    if isinstance(node, ast.UnaryOp):
        return _OPS[type(node.op)](_eval_expr(node.operand))
    raise ValueError("Unsupported")

@tool
def calculate(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '33 - 31'.

    Args:
        expression: The arithmetic expression to evaluate.
    """
    result = _eval_expr(ast.parse(expression, mode="eval").body)
    return str(result)


# Quick test
print("Weather tool:",    get_weather.invoke({"city": "Pune"}))
print("Calculator tool:", calculate.invoke({"expression": "33 - 31"}))

Weather tool: 31°C, clear sky
Calculator tool: 2


### The ReAct loop

The loop works like this:
```
1. Send the question to the model
2. If the model wants to call a tool → run it, send the result back
3. If the model gives a final answer → print it and stop
```

In [12]:
def run_tool(name, args, tools_map):
    """Run a tool by name and return its output as a string."""
    fn = tools_map.get(name)
    if fn is None:
        return f"Error: no tool called '{name}'."
    try:
        # LangChain @tool objects have an .invoke() method
        return str(fn.invoke(args))
    except Exception as e:
        return f"Tool error: {e}"


def react(question, tools_list, tools_map):
    """Simple ReAct loop — no memory yet."""

    # LangChain @tool objects wrap the real function in .func
    # ollama needs the raw function, not the wrapper
    raw_functions = [t.func for t in tools_list]

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": question},
    ]

    for step in range(8):  # max 8 steps to avoid infinite loops
        response = ollama.chat(model=MODEL, messages=messages, tools=raw_functions)
        messages.append(response.message)

        # No tool calls means the model has a final answer
        if not response.message.tool_calls:
            return response.message.content

        # Run each tool the model asked for
        for call in response.message.tool_calls:
            result = run_tool(call.function.name, call.function.arguments or {}, tools_map)
            print(f"  🔧 Tool called: {call.function.name} → {result}")
            messages.append({"role": "tool", "content": result})

    return "Reached the step limit without an answer."


# Test it!
tools_map_1 = {"get_weather": get_weather, "calculate": calculate}

answer = react(
    "Is it hotter in Pune or Mumbai? And by how many degrees?",
    [get_weather, calculate],
    tools_map_1,
)
print("\nAnswer:", answer)

  🔧 Tool called: get_weather → 31°C, clear sky
  🔧 Tool called: get_weather → 33°C, humid
  🔧 Tool called: calculate → 34

Answer: It is hotter in Mumbai compared to Pune by 3 degrees. The temperature in Mumbai is 33°C, while the temperature in Pune is 30°C (not 31 as previously mentioned).


**What you should see:** the agent calls `get_weather` for both cities, then `calculate` to find the difference, then gives you the final answer.

---
## Milestone 2 — Add a Knowledge Search Tool

The agent can answer weather and math questions. Now let's let it look up facts about the office.

We store the office facts in **ChromaDB**. This is the knowledge *base* (not memory — that comes in Milestone 3). The difference:

- **Knowledge base** = fixed facts that don't change (office hours, leave policy…)
- **Memory** = things the *user* told the agent during the conversation

### Store the office facts in ChromaDB

In [13]:
# These are the office facts we want the agent to know
OFFICE_FACTS = [
    "The Trinity office is open from 9am to 6pm on weekdays.",
    "Employees get 24 days of paid leave per year.",
    "The cafeteria serves lunch between 12pm and 2pm.",
    "Remote work is allowed up to three days per week.",
    "The IT helpdesk can be reached at extension 4500.",
]

# OllamaEmbeddings converts text into vectors that ChromaDB can store and search
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

# Chroma.from_texts() does two things:
#   1. Converts each fact into a vector using our embedding model
#   2. Stores everything in a ChromaDB collection called "office_knowledge"
knowledge_db = Chroma.from_texts(
    texts=OFFICE_FACTS,
    embedding=embeddings,
    collection_name="office_knowledge",
)

print(f"✅ Stored {len(OFFICE_FACTS)} facts in ChromaDB")

# Let's test the search directly to understand what's happening
results = knowledge_db.similarity_search("How many leave days?", k=2)
print("\n🔍 Test search for 'How many leave days?':")
for r in results:
    print(" →", r.page_content)

✅ Stored 5 facts in ChromaDB

🔍 Test search for 'How many leave days?':
 → Employees get 24 days of paid leave per year.
 → Remote work is allowed up to three days per week.


In [14]:
# Now wrap the ChromaDB search as a LangChain tool
knowledge_retriever = knowledge_db.as_retriever(search_kwargs={"k": 2})

@tool
def knowledge_search(query: str) -> str:
    """Look up facts about the Trinity office: hours, leave, remote work, cafeteria, IT.

    Args:
        query: What to look up, for example 'leave policy'.
    """
    # Retrieve the 2 most relevant facts from ChromaDB
    docs = knowledge_retriever.invoke(query)
    return " | ".join(doc.page_content for doc in docs)


# Test the tool
print("Knowledge tool test:", knowledge_search.invoke({"query": "remote work policy"}))

Knowledge tool test: Remote work is allowed up to three days per week. | Employees get 24 days of paid leave per year.


In [15]:
# Run the agent with the new knowledge tool added
tools_map_2 = {
    "get_weather":      get_weather,
    "calculate":        calculate,
    "knowledge_search": knowledge_search,
}

answer = react(
    "How many days of paid leave do employees get?",
    [get_weather, calculate, knowledge_search],
    tools_map_2,
)
print("\nAnswer:", answer)

  🔧 Tool called: knowledge_search → Employees get 24 days of paid leave per year. | Remote work is allowed up to three days per week.

Answer: According to our company's policies, employees are entitled to 24 days of paid leave per year, which can be used for various reasons such as vacation, illness, or personal events. Additionally, remote work is allowed up to three days per week, providing employees with flexibility and work-life balance.


**What you should see:** the agent calls `knowledge_search`, finds the leave fact, and answers "24 days".

Notice we didn't change `react()` at all — we just added one more tool to the list. That's the beauty of this design.

---
## Milestone 3 — ChromaDB Memory

Right now the agent forgets everything between questions. Ask it your name and it won't remember it on the next question.

### The old approach (dict + list)
```python
# This is what we had before
state = {"recent": [], "summary": "none yet"}
state["recent"].append({"role": "user", "content": question})
```
Problem: it grows forever, gets injected into every prompt, and disappears when the kernel restarts.

### The new approach (ChromaDB)
```
User says:  "My name is Priya"  →  embed → save to ChromaDB
User says:  "I work in Pune"    →  embed → save to ChromaDB
...
User asks:  "What city am I in?"  →  embed → search ChromaDB
                                      ↓
                         finds: "I work in Pune"  (similar meaning!)
                                      ↓
                         inject into prompt → agent answers "Pune"
```

### Step 3a — Create the memory

In [16]:
# Create a ChromaDB vector store just for conversation memory
# (This is separate from the knowledge_db above)
memory_db = Chroma(
    collection_name="conversation_memory",
    embedding_function=embeddings,
    persist_directory="./chroma_memory_db" # Forces disk persistence
)

# Wrap it as LangChain memory
# k=3 means: when the agent needs context, retrieve the 3 most relevant past turns
memory = VectorStoreRetrieverMemory(
    retriever=memory_db.as_retriever(search_kwargs={"k": 3}),
)

print("✅ ChromaDB memory created")
print("   Documents stored so far:", memory_db._collection.count())

✅ ChromaDB memory created
   Documents stored so far: 0


/tmp/ipykernel_211237/73714465.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  memory_db = Chroma(
/tmp/ipykernel_211237/73714465.py:11: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = VectorStoreRetrieverMemory(


### Step 3b — Understand how memory.save and memory.load work

Before building the full agent, let's see exactly what these two operations do.

In [17]:
# save_context() stores one conversation turn into ChromaDB
# It embeds the text and saves the vector + the original text
memory.save_context(
    {"input":  "My name is Priya and I manage the Pune team."},
    {"output": "Nice to meet you, Priya!"}
)
memory.save_context(
    {"input":  "I prefer Python over Java."},
    {"output": "Good choice!"}
)
memory.save_context(
    {"input":  "Our project deadline is next Friday."},
    {"output": "Understood."}
)

print(f"Stored {memory_db._collection.count()} turns in ChromaDB")
print()

# load_memory_variables() searches ChromaDB for the most relevant past turns
# It does NOT return all turns — only the ones semantically related to the query
relevant = memory.load_memory_variables({"prompt": "What city does the user work in?"})
print("🔍 Searching memory for: 'What city does the user work in?'")
print(relevant["history"])
print()

relevant2 = memory.load_memory_variables({"prompt": "What programming language does the user prefer?"})
print("🔍 Searching memory for: 'What programming language does the user prefer?'")
print(relevant2["history"])

Stored 3 turns in ChromaDB

🔍 Searching memory for: 'What city does the user work in?'
input: My name is Priya and I manage the Pune team.
output: Nice to meet you, Priya!
input: Our project deadline is next Friday.
output: Understood.
input: I prefer Python over Java.
output: Good choice!

🔍 Searching memory for: 'What programming language does the user prefer?'
input: I prefer Python over Java.
output: Good choice!
input: My name is Priya and I manage the Pune team.
output: Nice to meet you, Priya!
input: Our project deadline is next Friday.
output: Understood.


Notice that searching for "city" returns the Pune fact, not the Python fact. ChromaDB found the *relevant* memory without scanning everything — that's the key advantage.

### Step 3c — Build the agent with memory

In [18]:
def agent_with_memory(question, memory, tools_list, tools_map):
    """
    ReAct loop with ChromaDB memory.

    Two extra steps compared to react():
      BEFORE: load relevant memories from ChromaDB and add them to the system prompt
      AFTER:  save this turn to ChromaDB for future questions
    """

    # ── STEP 1: Load relevant memories ────────────────────────────────────────
    # ChromaDB searches for past turns that are similar to the current question
    past = memory.load_memory_variables({"prompt": question}).get("history", "")

    # Build the system prompt — inject memory if we have any
    if past:
        print(f"  💭 Memory retrieved: {str(past)[:100]}...")
        system_prompt = SYSTEM + f"\n\nWhat the user told you before:\n{past}"
    else:
        system_prompt = SYSTEM

    # ── STEP 2: Run the ReAct loop (same as before) ────────────────────────────
    raw_functions = [t.func for t in tools_list]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": question},
    ]

    answer = "Reached the step limit."
    for step in range(8):
        response = ollama.chat(model=MODEL, messages=messages, tools=raw_functions)
        messages.append(response.message)

        if not response.message.tool_calls:
            answer = response.message.content
            break

        for call in response.message.tool_calls:
            result = run_tool(call.function.name, call.function.arguments or {}, tools_map)
            print(f"  🔧 Tool: {call.function.name} → {result}")
            messages.append({"role": "tool", "content": result})

    # ── STEP 3: Save this turn to ChromaDB ────────────────────────────────────
    # This is what the old approach did with state["recent"].append(...)
    # Now it goes into ChromaDB as a vector instead of a plain list
    memory.save_context({"input": question}, {"output": answer})
    total = memory_db._collection.count()
    print(f"  💾 Turn saved. ChromaDB now has {total} turns stored.")

    return answer


print("✅ agent_with_memory() is ready")

✅ agent_with_memory() is ready


In [19]:
# Fresh memory for this test
test_memory_db = Chroma(
    collection_name="test_memory",
    embedding_function=embeddings,
)
test_memory = VectorStoreRetrieverMemory(
    retriever=test_memory_db.as_retriever(search_kwargs={"k": 3}),
)

tools_map_3 = {
    "get_weather":      get_weather,
    "calculate":        calculate,
    "knowledge_search": knowledge_search,
}
all_tools = [get_weather, calculate, knowledge_search]

print("=" * 55)
print("Turn 1 — introduce yourself")
print("=" * 55)
r1 = agent_with_memory(
    "My name is Priya and I manage the Pune team.",
    test_memory, all_tools, tools_map_3
)
print("Agent:", r1)

Turn 1 — introduce yourself
  🔧 Tool: knowledge_search → The Trinity office is open from 9am to 6pm on weekdays. | The cafeteria serves lunch between 12pm and 2pm.
  💾 Turn saved. ChromaDB now has 3 turns stored.
Agent: Hello Priya, welcome! I'd be happy to help you with any questions or concerns about the Pune team.

What can I assist you with today? Do you need information about a specific project, team member, or office policy? Or perhaps you have a question about the Trinity office in general?


In [20]:
print("=" * 55)
print("Turn 2 — test if it remembers name and city")
print("=" * 55)
r2 = agent_with_memory(
    "What is my name, and what is the weather where my team works?",
    test_memory, all_tools, tools_map_3
)
print("Agent:", r2)

Turn 2 — test if it remembers name and city
  💭 Memory retrieved: input: My name is Priya and I manage the Pune team.
output: Hello Priya, welcome! I'd be happy to he...
  🔧 Tool: get_weather → 31°C, clear sky
  💾 Turn saved. ChromaDB now has 3 turns stored.
Agent: Hello Priya, welcome! I'd be happy to help you with any questions or concerns about the Pune team.

It looks like the weather in Pune is currently 31 degrees Celsius and mostly sunny. Hopefully, this will make for a pleasant working day! Is there anything else I can assist you with regarding your team or office?


In [22]:
!pip install langchain-community


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# Summary Memory

## What is Summary Memory?

Summary Memory is a memory management technique where an LLM continuously summarizes the conversation instead of storing every message.

Instead of keeping the entire chat history, the model maintains a concise summary that captures the most important information from previous interactions.

---

## Why Do We Need It?

Imagine the following conversation:

User: My name is Priya.  
User: I work in Pune.  
User: I manage a team of 10 engineers.  
User: My project deadline is next Friday.

If we keep storing every message forever, the conversation history becomes very large and expensive to send to the model.

Summary Memory solves this problem by creating a condensed summary such as:

> Priya works in Pune, manages a team of 10 engineers, and has a project deadline next Friday.

The summary replaces many older messages while preserving the main context.

---

## How It Works

1. Store conversation turns.
2. Send conversation history to an LLM.
3. Generate a concise summary.
4. Update the summary whenever new messages arrive.
5. Use the summary as context for future responses.

---

## Advantages

- Reduces context size.
- Saves tokens and computation cost.
- Suitable for long conversations.
- Easy to implement.

---

## Limitations

- Important details may be lost.
- Summary quality depends on the LLM.
- Exact wording is not preserved.

Example:

Original:
> My meeting is on Friday at 3 PM.

Summary:
> My meeting is on Friday.

The time information may be lost during summarization.

---

## When to Use Summary Memory

Summary Memory is useful when:

- Conversations are very long.
- Exact wording is not critical.
- Reducing token usage is important.

Examples:

- Personal assistants
- Customer support chatbots
- Long-running AI conversations

---

## Comparison with Other Memory Types

| Memory Type | Stores |
|------------|---------|
| Sliding Window Memory | Recent N messages |
| Summary Memory | Conversation summary |
| Vector Memory | Semantic embeddings |

Summary Memory compresses information, while Vector Memory retrieves relevant information when needed.

In [24]:
# SUMMARY MEMORY

from langchain_classic.memory import ConversationSummaryMemory
from langchain_ollama import ChatOllama

# LLM used to generate summaries
summary_llm = ChatOllama(
    model=MODEL,
    temperature=0
)

# Summary Memory
summary_memory = ConversationSummaryMemory(
    llm=summary_llm,
    return_messages=True
)

# Simulate a conversation
summary_memory.save_context(
    {"input": "My name is Priya and I work in Pune."},
    {"output": "Nice to meet you Priya."}
)

summary_memory.save_context(
    {"input": "I manage a team of 10 engineers."},
    {"output": "That sounds interesting."}
)

summary_memory.save_context(
    {"input": "Our project deadline is next Friday."},
    {"output": "Got it."}
)

# View generated summary
print("Generated Summary:\n")
print(summary_memory.buffer)

/tmp/ipykernel_211237/199003466.py:13: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  summary_memory = ConversationSummaryMemory(


Generated Summary:

The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential. The human then introduces themselves, stating that their name is Priya and they work in Pune. The AI responds with a friendly greeting, saying "Nice to meet you Priya." Priya shares some additional information about herself by mentioning that she manages a team of 10 engineers. The conversation takes a practical turn as Priya mentions the project deadline next Friday, which the AI acknowledges with a simple "Got it."


# LangChain Runnables and the Pipe (`|`) Operator

## What is a Runnable?

A Runnable is any LangChain component that can receive input and produce output.

Examples:

- Prompt Templates
- Language Models (LLMs)
- Retrievers
- Output Parsers
- Custom Functions

All of these follow a common interface:

- `.invoke()`
- `.batch()`
- `.stream()`

---

## Runnable Pipeline

LangChain allows us to connect multiple Runnable components together using the pipe (`|`) operator.

Example:

```python
chain = prompt | llm | parser

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL)

prompt = ChatPromptTemplate.from_template(
    """
    You are an AI tutor.

    Explain the following concept in simple terms:

    {topic}
    """
)

parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke(
    {"topic": "ChromaDB"}
)

print(response)

I'd be happy to explain ChromaDB in simple terms.

ChromaDB is a free, open-source music cataloging tool that helps you organize and manage your digital music collection. Here's how it works:

**What does "Chroma" mean?**
The name "ChromaDB" comes from the Greek word "chroma," which means color. In music, chroma refers to the color or tone of a sound.

**What is ChromaDB?**
ChromaDB is a database that stores information about your digital music files, such as their title, artist, album, genre, and more. It's like a virtual folder for your music collection.

**How does it work?**

1. You add your digital music files to ChromaDB by importing them from various sources, such as iTunes, Spotify, or Google Play Music.
2. The tool analyzes the metadata (information) associated with each file and creates a detailed profile of each song.
3. You can then search, sort, and categorize your songs using tags like genre, mood, or artist.

**Key features:**

* Tag-based organization: Use keywords to l

**What you should see:**
- Turn 1: the agent greets Priya and saves the turn to ChromaDB
- Turn 2: the `💭 Memory retrieved` line shows ChromaDB found the Pune/Priya fact
- The agent answers with Priya's name AND calls `get_weather` for Pune — tools + memory together!

---
## What changed from the old dict+list approach?

| | Old (Day 4) | New (Today) |
|---|---|---|
| **Store a turn** | `state["recent"].append(...)` | `memory.save_context(...)` → ChromaDB |
| **Load context** | Inject ALL recent turns | Search and inject RELEVANT turns only |
| **Survives restart** | ❌ No | ✅ Yes (if you use `persist_directory`) |
| **Works at scale** | Gets slow after ~50 turns | Fast even with thousands of turns |

---
## Recap — what we built

```
User question
     ↓
Search ChromaDB memory  ← relevant past turns
     ↓
Build system prompt (SYSTEM + memory)
     ↓
ReAct loop
  ├── call get_weather   (tool)
  ├── call calculate     (tool)
  └── call knowledge_search → ChromaDB knowledge base
     ↓
Final answer
     ↓
Save turn to ChromaDB memory
```

### Try it yourself
1. Tell the agent your favourite city, then ask for its weather in the next message
2. Tell the agent you have a deadline on Monday, then ask "when is my deadline?" later
3. Ask about both leave policy (knowledge base) and your personal info (memory) in one question